In [8]:
import cv2
import numpy as np
import os

# Example of using SIFT to extract keypoints and descriptors
def extract_features(image, detector='SIFT'):
    if detector == 'SIFT':
        sift = cv2.SIFT_create()
        keypoints, descriptors = sift.detectAndCompute(image, None)
    elif detector == 'SURF':
        surf = cv2.xfeatures2d.SURF_create()
        keypoints, descriptors = surf.detectAndCompute(image, None)
    elif detector == 'ORB':
        orb = cv2.ORB_create()
        keypoints, descriptors = orb.detectAndCompute(image, None)
    elif detector == 'AKAZE':
        akaze = cv2.AKAZE_create()
        keypoints, descriptors = akaze.detectAndCompute(image, None)
    
    return descriptors

from sklearn.cluster import KMeans
from scipy.cluster.vq import vq

# Create a vocabulary of visual words using KMeans clustering
def build_vocabulary(descriptors_list, num_clusters=100):
    all_descriptors = np.vstack(descriptors_list)
    kmeans = KMeans(n_clusters=num_clusters)
    kmeans.fit(all_descriptors)
    return kmeans

# Convert descriptors into histograms using the vocabulary (BoW model)
def compute_histograms(descriptors, kmeans):
    # Assign each descriptor to a cluster (visual word)
    words, _ = vq(descriptors, kmeans.cluster_centers_)
    
    # Create a histogram of visual words
    histogram, _ = np.histogram(words, bins=np.arange(len(kmeans.cluster_centers_)) + 1)
    return histogram

# Load images and extract features
def prepare_dataset(images, detector='SIFT', num_clusters=100):
    descriptors_list = []
    histograms = []
    
    for img in images:
        descriptors = extract_features(img, detector=detector)
        if descriptors is not None:
            descriptors_list.append(descriptors)
    
    # Build the BoW model
    kmeans = build_vocabulary(descriptors_list, num_clusters=num_clusters)
    
    # Compute histograms for each image
    for img in images:
        descriptors = extract_features(img, detector=detector)
        if descriptors is not None:
            histogram = compute_histograms(descriptors, kmeans)
            histograms.append(histogram)
    
    return np.array(histograms)

# Example images list
images = []

PATH = 'archive/GarbageClassification/cardboard/'

for img in os.listdir(PATH):
    images.append(cv2.imread(PATH+img, cv2.IMREAD_GRAYSCALE))
# for img in images:
#     cv2.imshow('Image', img)
#     cv2.waitKey(0)
# Prepare the dataset
X = prepare_dataset(images, detector='SIFT', num_clusters=100)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split

# Assuming y contains the corresponding labels
y = []
for i in range(0, 200):
    y.append(0)
    y.append(1)

y = np.array(y)
# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Build a simple fully connected network
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')  # Adjust output size for your classes
])

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, epochs=10, batch_size=32)

# Evaluate the model
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test accuracy: {test_acc}")


Epoch 1/10


c:\Users\steve\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.1901 - loss: 5.5765
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5433 - loss: 1.4271
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5618 - loss: 0.9500
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6870 - loss: 0.6944
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 893us/step - accuracy: 0.7541 - loss: 0.5988
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7678 - loss: 0.5387
Epoch 7/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8219 - loss: 0.5091  
Epoch 8/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8796 - loss: 0.4201
Epoch 9/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8846 - loss: 0.3762
Epoch 10/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 934us/step - accuracy: 0.9022 - loss: 0.3385
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5500 - loss: 0.8417
Test accuracy: 0.5375000238418579
